# P5-4. 최종 미니프로젝트 — 딥러닝 모델링 — 코드 구현 가이드 (TensorFlow)

**주제: 고객 이탈(Churn) 예측 — DNN 모델 만들기**

## 수업 목적
P5-3과 동일한 고객 이탈 데이터를 사용하여 TensorFlow/Keras 기반 DNN을 직접 구현한다.
완성 코드를 복사하기보다 **Train/Validation/Test 분리, DNN 구조, Callback, 클래스 불균형 대응, 최종 평가**를 재구성하는 데 목적이 있다.

## 핵심 구현 흐름
`데이터 준비 → Train/Test → Train/Validation → 스케일링 → DNN → Callback → 학습 → 과적합 진단 → Test 평가 → class_weight → ML vs DL 비교`

## 구현 원칙
- P5-3과 같은 `test_size=0.30`, `random_state=42`를 사용한다.
- Test는 마지막 평가에서만 사용한다.
- Validation은 Train에서 별도로 분리한다.
- 이진분류이므로 출력층은 `Dense(1, sigmoid)`를 사용한다.
- 손실함수는 `binary_crossentropy`를 사용한다.

## 0. 라이브러리 준비
TensorFlow, pandas, numpy, matplotlib와 scikit-learn의 분할·스케일링·평가 도구를 불러온다.

In [ ]:
# 필요한 라이브러리와 RANDOM_STATE를 직접 작성한다.

## 1. 데이터 로드
`data_save.csv`를 읽고 데이터 크기를 확인한다.

In [ ]:
# 데이터를 불러온다.

## 2. 전처리 — P5-3과 동일한 방식
문자열형 컬럼을 One-Hot Encoding하고 X와 y를 분리한다.

In [ ]:
# One-Hot Encoding 후 X와 y를 분리한다.

## 3. Train/Test 분할 — P5-3과 동일 조건
Test 30%, `stratify=y`, `random_state=42`를 사용한다.

In [ ]:
# Train 전체와 Test를 분리한다.

## 4. Train/Validation 분할
Train 전체 데이터에서 Validation 20%를 추가로 분리한다.

**중요:** `validation_data=(X_test, y_test)`로 두지 않는다.
Test가 EarlyStopping이나 모델 선택에 사용되면 최종 평가 데이터의 독립성이 깨진다.

In [ ]:
# Train과 Validation을 분리한다.

## 5. 스케일링
Train에 `fit_transform()`, Validation/Test에는 `transform()`만 적용한다.

In [ ]:
# MinMaxScaler로 세 데이터셋을 변환한다.

## 6. DNN 모델 정의
다음 구조를 직접 구현한다.

`입력 → Dense(64, ReLU) → Dropout(0.3) → Dense(32, ReLU) → Dropout(0.3) → Dense(1, Sigmoid)`

컴파일 조건:
- Optimizer: Adam, learning rate 0.001
- Loss: binary_crossentropy
- Metric: accuracy

모델 생성을 함수로 만들면 baseline과 class_weight 모델을 같은 구조로 재사용할 수 있다.

In [ ]:
# build_dnn(input_dim) 함수를 구현하고 model.summary()를 확인한다.

## 7. Callback 구성
다음 두 Callback을 구성한다.

1. `EarlyStopping`
   - `monitor='val_loss'`
   - `patience=5`
   - 최적 가중치 복원

2. `ModelCheckpoint`
   - Validation loss가 가장 좋은 모델만 저장

In [ ]:
# EarlyStopping과 ModelCheckpoint를 구성한다.

## 8. 기본 DNN 학습
Train으로 학습하고 Validation으로 매 epoch 성능을 확인한다.

권장값: `epochs=50`, `batch_size=32`

In [ ]:
# model.fit()을 작성한다.

## 9. 학습곡선으로 과적합 진단
Train/Validation의 Loss와 Accuracy를 각각 그래프로 비교한다.

In [ ]:
# history.history를 DataFrame으로 바꾸고 학습곡선을 그린다.

## 10. 공통 평가 함수
DNN은 확률을 출력하므로 다음 흐름이 필요하다.

`예측 확률 → threshold 0.5 적용 → 0/1 클래스 변환 → 지표 계산`

Accuracy, Precision, Recall, F1을 반환하는 평가 함수를 구현한다.

In [ ]:
# evaluate_dnn() 함수를 구현한다.

## 11. 기본 DNN 최종 Test 평가
학습 중 사용하지 않은 Test 데이터로 baseline DNN을 한 번 평가한다.

In [ ]:
# baseline 모델을 Test 데이터로 평가한다.

## 12. 클래스 불균형 대응 — class_weight
`compute_class_weight(class_weight='balanced', ...)`를 사용하여 클래스별 가중치를 계산하고
Keras가 요구하는 `{클래스번호: 가중치}` 딕셔너리로 변환한다.

In [ ]:
# class_weight를 계산한다.

## 13. class_weight 적용 DNN 재학습
같은 구조의 새 DNN 모델을 만든 뒤 `model.fit(..., class_weight=class_weights)`로 학습한다.

**주의:** baseline 모델을 이어서 학습하지 않고 새 모델을 생성한다.
그래야 class_weight 적용 전후를 공정하게 비교할 수 있다.

In [ ]:
# 새 모델과 Callback을 만들고 class_weight를 적용하여 학습한다.

## 14. class_weight 적용 DNN 평가
Test 데이터에서 baseline과 weighted DNN의 성능을 비교하고 classification report와 confusion matrix를 확인한다.

In [ ]:
# weighted DNN을 평가하고 두 결과를 DataFrame으로 비교한다.

## 15. 머신러닝 vs 딥러닝 비교
P5-3의 대표 머신러닝 모델과 DNN을 같은 지표로 비교한다.

| 구분 | 대표 모델 | Accuracy | Precision | Recall | F1 |
|---|---|---:|---:|---:|---:|
| 머신러닝 | P5-3 선택 모델 |  |  |  |  |
| 딥러닝 | baseline 또는 class_weight |  |  |  |  |

## 최종 점검표
| 확인 항목 | 핵심 기준 |
|---|---|
| Test 분리 | P5-3과 동일한 조건인가 |
| Validation | Train에서 별도로 분리했는가 |
| 스케일링 | Train에만 fit했는가 |
| 출력층 | 이진분류에 맞는 sigmoid 1개인가 |
| Loss | binary_crossentropy인가 |
| Callback | EarlyStopping과 Checkpoint를 적용했는가 |
| Test 사용 | 학습 중 Test를 사용하지 않았는가 |
| 불균형 대응 | class_weight 전후를 비교했는가 |
| 평가 | Accuracy, Precision, Recall, F1을 비교했는가 |
| 해석 | ML과 DL의 성능·복잡도를 함께 비교했는가 |